In [7]:
import pandas as pd
from pathlib import Path

cub_root = Path("D:/Princeton/Classes/COS 429/cv_emergence_project/data/CUB_200_2011")
meta_dir = cub_root / "metadata"

images_df = pd.read_csv(meta_dir / "images.csv")
classes_df = pd.read_csv(meta_dir / "classes.csv")
parts_df = pd.read_csv(meta_dir / "parts.csv")

images_df.head(), classes_df.head(), parts_df.head()

(   image_id                                          file_path  class_id  \
 0         1  001.Black_footed_Albatross/Black_Footed_Albatr...         1   
 1         2  001.Black_footed_Albatross/Black_Footed_Albatr...         1   
 2         3  001.Black_footed_Albatross/Black_Footed_Albatr...         1   
 3         4  001.Black_footed_Albatross/Black_Footed_Albatr...         1   
 4         5  001.Black_footed_Albatross/Black_Footed_Albatr...         1   
 
                    class_name  is_train  bbox_x  bbox_y  bbox_width  \
 0  001.Black_footed_Albatross         0    60.0    27.0       325.0   
 1  001.Black_footed_Albatross         1   139.0    30.0       153.0   
 2  001.Black_footed_Albatross         0    14.0   112.0       388.0   
 3  001.Black_footed_Albatross         1   112.0    90.0       255.0   
 4  001.Black_footed_Albatross         1    70.0    50.0       134.0   
 
    bbox_height  
 0        304.0  
 1        264.0  
 2        186.0  
 3        242.0  
 4        30

In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader
from datasets.cub_dataset import CUBDataset

cub_root = "data/CUB_200_2011/"

tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

train_ds = CUBDataset(cub_root, split="train", transform=tf)
len(train_ds), train_ds[0]["image"].shape, train_ds[0]["label"]


ModuleNotFoundError: No module named 'torchvision'

In [ ]:
import torch
from torchvision import transforms
from torch.utils.data import DataLoader
from datasets.cub_dataset import CUBDataset
from models.resnet_cub import get_resnet50_cub, load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cub_root = "data/CUB_200_2011/CUB_200_2011"

tf_test = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

test_ds = CUBDataset(cub_root, split="test", transform=tf_test)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

model = get_resnet50_cub()
model = load_checkpoint(model, "checkpoints/resnet50_cub_best.pth", device=device).to(device)
model.eval()

correct, total = 0, 0
with torch.no_grad():
    for batch in test_loader:
        imgs, labels = batch["image"].to(device), batch["label"].to(device)
        logits = model(imgs)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)

print("Test accuracy:", correct / total)


In [ ]:
from pathlib import Path
import torch

feat_dir = Path("features/resnet50_cub")

conv1_train = torch.load(feat_dir / "conv1_train.pt")
layer4_train = torch.load(feat_dir / "layer4_train.pt")
labels = torch.load(feat_dir / "labels_train.pt")["labels"]

conv1_train.shape, layer4_train.shape, labels.shape


In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

with open("results/probes/resnet50_cub_probes.json", "r") as f:
    res = json.load(f)

df = pd.DataFrame(res)

df_subclass = df[df["target_type"] == "subclass"]
df_concepts = df[df["target_type"] == "concept"]

plt.figure()
plt.plot(df_subclass["layer"], df_subclass["val_acc"], marker="o", label="Species (subclass)")
plt.xticks(rotation=45)
plt.ylabel("Validation accuracy")
plt.title("Emergence of species information across layers")
plt.legend()
plt.tight_layout()
plt.show()

# For one concept (e.g., first in df_concepts)
if len(df_concepts) > 0:
    concept_name = df_concepts["target_name"].iloc[0]
    df_c = df_concepts[df_concepts["target_name"] == concept_name]
    plt.figure()
    plt.plot(df_c["layer"], df_c["val_acc"], marker="o", label=concept_name)
    plt.xticks(rotation=45)
    plt.ylabel("Validation accuracy")
    plt.title(f"Emergence of concept {concept_name} across layers")
    plt.legend()
    plt.tight_layout()
    plt.show()
